# Traducteur français → duala (NLLB-200 affiné) — Safinum / AfroLuo
**Colab : Exécution → Modifier le type d'exécution → GPU T4.** Exécuter les cellules dans l'ordre.

Principe (honnête) : le dictionnaire apprend le **vocabulaire**, pas la **construction de phrases**.
Le pipeline combine donc : (1) variables `{montant}` protégées, (2) glossaire exact, (3) modèle NLLB affiné,
(4) contrôle automatique des variables, (5) remplacement des mots inconnus par un synonyme présent dans le dictionnaire, (6) indicateurs « à relire ». Les phrases du site ne seront bonnes que si des phrases **validées par un locuteur natif**
(colonne « Traduction duala » du xlsx, Statut = Validé) sont ajoutées à l'entraînement — la cellule 3 les intègre.

In [2]:
!pip -q install transformers accelerate sacrebleu sentencepiece openpyxl nltk
import nltk
for _p in ('wordnet', 'omw-1.4', 'omw-2.0'):          # WordNet français : sert à trouver des synonymes (cellule 7d)
    try: nltk.download(_p, quiet=True)
    except Exception as e: print('nltk', _p, e)
import torch; print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN -> activer un GPU')


GPU : Tesla T4


## 1. Configuration

In [3]:
import re, random, unicodedata, collections, json, os, math, time
import numpy as np, pandas as pd, torch, sacrebleu

SEED = 42
MODEL = 'facebook/nllb-200-distilled-600M'
SRC, TGT = 'fra_Latn', 'lin_Latn'   # NLLB n'a pas le duala : on réutilise le code d'une langue bantoue (lingala) comme étiquette. Aucun impact sur la qualité.
STRIP_TONES = True                  # retire les accents de ton (grave, aigu, circonflexe, macron, caron) : 63 % des mots en ont, trop de dispersion pour 25 k paires
RESTORE_TONES = True               # remet les tons en sortie (glossaire + phrases natives), cf. section 7
FREEZE_EMB  = True                  # fige les embeddings : moins de sur-apprentissage, moins de mémoire
MAX_WORDS_DICO = 8                  # seuil mot/phrase : au-delà de 8 mots, une paire « ok » du dictionnaire est traitée comme une PHRASE native (conservée, répétée VALIDATED_UPSAMPLE fois)
RATIO_ALIGN = 3.0                   # une phrase longue est écartée seulement si sa traduction est > 3 fois plus longue/courte (ex. 1 mot en face de 11 mots) ; None = ne rien écarter
MAX_WORDS_PHRASES = 80              # phrases natives validées : nombre de mots maximal conservé (français et duala)
MAX_LEN = 160                       # tokens max (source et cible). Une phrase plus longue est découpée à la traduction
NO_REPEAT = 3                       # interdit de répéter un même groupe de 3 tokens ; si de longues sorties semblent tronquées, tester 0 (désactivé) ou 4
SYNONYMES = True                    # mot absent du dictionnaire -> synonyme présent dans le dictionnaire (cellule 7d)
MIN_SYN = 0.9                       # seuil de confiance WordNet : 0.9 ~ synonyme du sens principal seulement (0.34 acceptait « dépôt -> entrepôt »)
EPOCHS, PATIENCE, BS, LR, LS = 14, 3, 32, 1e-4, 0.1
MAX_MINUTES = 100                  # l'entraînement s'arrête proprement après cette durée ; le meilleur modèle est conservé
EVAL_N, EVAL_BEAMS = 500, 1        # validation à chaque époque : 500 termes, décodage rapide
SAVE_EVERY, CKPT_DIR = 4, 'best'   # écriture disque toutes les 4 époques ; mettre un dossier Drive pour survivre à une déconnexion
DROPOUT = 0.2                       # régularisation (NLLB : 0.1 par défaut) ; monter à 0.3 si l'écart train/val reste > 1.2
VALIDATED_UPSAMPLE = 3              # poids des phrases validées par un natif
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Fichiers
Téléverser `dico_paires_nettoyees.csv` (obligatoire) et `phrases_duala_nettoye.xlsx` rempli par les traducteurs (facultatif au premier essai).

In [4]:
import os
import glob
try:
    from google.colab import files
    up = files.upload()           # sélectionner les 1 ou 2 fichiers
except ImportError:
    files = None
PAIRS_CSV = 'dico_paires_nettoyees.csv'
# n'importe quel .xlsx téléversé est accepté (le nom exact n'a pas d'importance)
_xl = [f for f in glob.glob('*.xlsx') if not f.startswith('~$') and 'avec_brouillon' not in f]
SENT_XLSX = max(_xl, key=os.path.getmtime) if _xl else None
print('CSV dictionnaire :', PAIRS_CSV, os.path.exists(PAIRS_CSV))
print('XLSX phrases     :', SENT_XLSX if SENT_XLSX else 'AUCUN (le modèle ne sera entraîné que sur le dictionnaire)')
if len(_xl) > 1: print('⚠ plusieurs xlsx trouvés, le plus récent est utilisé :', _xl)


Saving dico_paires_nettoyees.csv to dico_paires_nettoyees.csv
CSV dictionnaire : dico_paires_nettoyees.csv True
XLSX phrases     : AUCUN (le modèle ne sera entraîné que sur le dictionnaire)


## 3. Préparation : normalisation, variables protégées, découpage sans fuite

In [5]:
VAR = re.compile(r'\{[^{}]*\}')
TONE = {c: None for c in (0x300, 0x301, 0x302, 0x304, 0x30c)}

def norm_dua(s):
    s = unicodedata.normalize('NFD', str(s).replace('’', "'").replace('‘', "'").replace('–', '-'))
    if STRIP_TONES: s = s.translate(TONE)
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', s)).strip()
def norm_fr(s):
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(s))).strip()

def protect(s):
    """{montant} -> [[0]] ; retourne texte et liste des variables dans l'ordre."""
    vs = []
    def f(m): vs.append(m.group(0)); return f'[[{len(vs)-1}]]'
    return VAR.sub(f, s), vs
def restore(s, vs):
    for i, v in enumerate(vs): s = s.replace(f'[[{i}]]', v)
    return s
def same_vars(a, b): return collections.Counter(VAR.findall(a)) == collections.Counter(VAR.findall(b))

# --- dictionnaire (paires jugées fiables) ---
d = pd.read_csv(PAIRS_CSV)
d = d[d.qualite == 'ok'].copy()      # seul le statut « ok » est utilisé (« a_verifier » reste exclu)
_FRSW = set('le les des du de et vous votre vos est sont pour avec dans une un'.split())
def _nfw(t): return sum(w in _FRSW for w in re.findall(r"[^\W\d_]+", str(t).lower()))
_sw = (d.duala.map(_nfw) >= 3) & (d.francais.map(_nfw) == 0)      # colonnes inversées (français dans « duala »)
if _sw.any():
    print(f'{_sw.sum()} ligne(s) avec colonnes duala/français inversées : remises dans le bon sens')
    d.loc[_sw, ['duala', 'francais']] = d.loc[_sw, ['francais', 'duala']].values
d['fr'] = d.francais.map(norm_fr); d['dua'] = d.duala.map(norm_dua)
_nf, _nd = d.fr.str.split().str.len(), d.dua.str.split().str.len()
_long = (_nf > MAX_WORDS_DICO) | (_nd > MAX_WORDS_DICO)
_r = _nd.clip(lower=1) / _nf.clip(lower=1)
_mal = (_long & ((_r > RATIO_ALIGN) | (_r < 1 / RATIO_ALIGN))) if RATIO_ALIGN else (_long & False)
_tropl = (_nf > MAX_WORDS_PHRASES) | (_nd > MAX_WORDS_PHRASES)
print(f'dictionnaire (ok) : {(_long & ~_mal & ~_tropl).sum()} phrases de plus de {MAX_WORDS_DICO} mots CONSERVÉES (traitées comme phrases natives) | {_mal.sum()} écartées (longueurs incompatibles) | {_tropl.sum()} de plus de {MAX_WORDS_PHRASES} mots')
if _mal.any(): print('  écartées car longueurs incompatibles (ex. un mot en face de 11 mots) :'); print(d[_mal][['francais', 'duala']].head(15).to_string(max_colwidth=60))
d_long = d[_long & ~_mal & ~_tropl].copy()
d = d[~_long & (d.fr.str.len() > 1) & (d.dua.str.len() > 1)]
d = d.drop_duplicates(['fr', 'dua'])
d['key'] = d.fr.str.lower()
print(len(d), 'paires,', d.key.nunique(), 'termes français distincts')

# --- découpage PAR TERME FRANÇAIS : aucun terme du test/val n'apparaît dans l'entraînement ---
keys = sorted(d.key.unique()); random.Random(SEED).shuffle(keys)
n = len(keys); K = {'test': set(keys[:n//20]), 'val': set(keys[n//20: n//10])}
split = d.key.map(lambda k: 'test' if k in K['test'] else 'val' if k in K['val'] else 'train')
train, val, test = (d[split == s] for s in ('train', 'val', 'test'))
print('train/val/test :', len(train), len(val), len(test))

# --- phrases validées par un natif (xlsx) ---
def _n(s):  # texte comparable : sans accents, minuscules, sans espaces autour
    return unicodedata.normalize('NFD', str(s)).encode('ascii', 'ignore').decode().lower().strip()

sent_tr, sent_te, sent_va, native_raw = [], [], [], {}
FR_COL, sheet = 4, 'Phrases'
if SENT_XLSX:
    xl = pd.ExcelFile(SENT_XLSX)
    sheet = 'Phrases' if 'Phrases' in xl.sheet_names else xl.sheet_names[0]
    x = xl.parse(sheet)
    print(f'xlsx : feuille « {sheet} », {x.shape[0]} lignes, {x.shape[1]} colonnes')
    for i, c in enumerate(x.columns): print(f'  colonne {i+1} : {c!r}  ({x[c].notna().sum()} cellules remplies)')
    cols = list(x.columns)
    # colonnes retrouvées par leur EN-TÊTE ; à défaut, par leur position historique
    def find(*words, default=None):
        for c in cols:
            if any(w in _n(c) for w in words): return c
        return cols[default] if default is not None and default < len(cols) else None
    c_id, c_prio = find('id', default=0), find('prio', default=1)
    c_fr = find('franc', 'source', default=3)
    c_stat = find('statut', 'status', default=8)
    c_dua = [c for c in cols if 'duala' in _n(c) or 'traduction' in _n(c) or _n(c) in ('dua', 'trad')]
    c_dua = [c for c in c_dua if 'brouillon' not in _n(c) and 'modele' not in _n(c)]
    if not c_dua: c_dua = [cols[i] for i in (4, 9) if i < len(cols)]
    FR_COL = cols.index(c_fr) + 1 if c_fr in cols else 4
    print('colonnes utilisées -> français :', c_fr, '| statut :', c_stat, '| duala :', c_dua, '| priorité :', c_prio)
    x = x.copy()
    x['_fr'] = x[c_fr]
    x['_dua'] = x[c_dua].bfill(axis=1).iloc[:, 0] if c_dua else np.nan      # 1re colonne duala remplie
    x['_stat'] = x[c_stat].map(_n) if c_stat else ''
    x['_prio'] = pd.to_numeric(x[c_prio].astype(str).str.extract(r'(\d+)')[0], errors='coerce') if c_prio else np.nan
    print('valeurs de la colonne Statut :', dict(x['_stat'].value_counts(dropna=False)))
    ok_stat = x['_stat'].str.startswith('valid') & ~x['_stat'].str.startswith('non')
    ok_txt = x['_dua'].notna() & x['_fr'].notna() & (x['_dua'].astype(str).str.strip() != '')
    print(f'lignes Validé : {ok_stat.sum()} | duala rempli : {ok_txt.sum()} | les deux : {(ok_stat & ok_txt).sum()}')
    x = x[ok_stat & ok_txt]
    bad = x[~x.apply(lambda r: same_vars(str(r._fr), str(r._dua)), axis=1)]
    if len(bad): print('IGNORÉES (variables différentes entre français et duala) :', len(bad), 'lignes ->', list(bad[c_id])[:20])
    x = x.drop(bad.index)
    # longueur des phrases natives : on les garde LONGUES (jusqu'à MAX_WORDS_PHRASES mots), contrairement au dictionnaire
    if len(x):
        nw = x._fr.astype(str).str.split().str.len(); nd = x._dua.astype(str).str.split().str.len()
        print(f'phrases natives : médiane {nw.median():.0f} mots (français), max {nw.max()} | > {MAX_WORDS_DICO} mots : {(nw > MAX_WORDS_DICO).sum()}/{len(x)}')
        too_long = (nw > MAX_WORDS_PHRASES) | (nd > MAX_WORDS_PHRASES)
        if too_long.any(): print(f'  {too_long.sum()} phrases dépassent {MAX_WORDS_PHRASES} mots : ignorées (monter MAX_WORDS_PHRASES ou les couper en deux)')
        x = x[~too_long]
    # jeu de test « site » : 20 % des phrases de priorité 1 mises de côté (jamais entraînées)
    p1 = x[x._prio == 1].sample(frac=0.2, random_state=SEED) if (x._prio == 1).any() else x.iloc[:0]
    # 10 % des phrases restantes servent à choisir la meilleure époque (validation) ; le test « site » reste intact
    _rest = x.drop(p1.index)
    va = _rest.sample(frac=0.1, random_state=SEED) if len(x) >= 30 else _rest.iloc[:0]
    for idx, r in x.iterrows():
        native_raw[norm_fr(r._fr).lower()] = norm_fr(r._dua)      # forme écrite par le locuteur, tons conservés
        pair = (norm_fr(r._fr), norm_dua(r._dua))
        (sent_te if idx in p1.index else sent_va if idx in va.index else sent_tr).append(pair)
    print('phrases validées : train', len(sent_tr), '| validation', len(sent_va), '| test site', len(sent_te))
    if not (sent_tr or sent_te):
        print('⚠ 0 phrase retenue : lire les lignes « valeurs de la colonne Statut » et « lignes Validé / duala rempli » ci-dessus.')
else:
    print('Pas de xlsx : entraînement sur le dictionnaire seul.')

# --- phrases longues du dictionnaire (statut ok) : traitées comme des phrases natives validées ---
if len(d_long):
    dl = d_long.sample(frac=1, random_state=SEED); k_ = len(dl) // 10 if len(dl) >= 20 else 0
    for j, (_, r) in enumerate(dl.iterrows()):
        key_ = r.fr.lower()
        if key_ in native_raw: continue                                  # le xlsx est prioritaire
        native_raw[key_] = norm_fr(r.duala)                              # forme avec tons, pour la réponse exacte
        (sent_te if j < k_ else sent_va if j < 2 * k_ else sent_tr).append((r.fr, r.dua))
    print(f'phrases longues du dictionnaire : {len(dl)} ajoutées -> total phrases natives : train {len(sent_tr)} | validation {len(sent_va)} | test {len(sent_te)}')
if len(sent_tr) < 200: print('⚠ moins de 200 phrases natives d’entraînement : le dictionnaire apprend le vocabulaire, pas la construction des phrases ; les phrases longues resteront approximatives.')

def mk(pairs):  # liste (fr, dua) -> exemples avec variables protégées
    out = []
    for fr, du in pairs:
        f, vs = protect(fr); g, _ = protect(du); out.append((f, g))
    return out
# --- table de restauration des tons : mot sans ton -> forme tonale ---
# On ne garde une forme que si elle est NON AMBIGUË (>= 80 % des occurrences) ; sinon le mot reste sans ton plutôt que faux.
def build_tone_map(duala_texts, native_texts, min_share=0.8):
    cnt = collections.defaultdict(collections.Counter)
    for t in duala_texts:
        for w in str(t).split(): cnt[norm_dua(w).lower()][unicodedata.normalize('NFC', w).lower()] += 1
    tm = {}
    for k, c in cnt.items():
        form, n_ = c.most_common(1)[0]
        if form != k and n_ / sum(c.values()) >= min_share: tm[k] = form
    for t in native_texts:                                          # les natifs priment sur le dictionnaire
        for w in t.split():
            tm[norm_dua(w).lower()] = unicodedata.normalize('NFC', w).lower()
    return tm
tone_map = build_tone_map(d.duala, native_raw.values())
def restore_tones(text):
    if not RESTORE_TONES: return text
    def f(m):
        w = m.group(0); t = tone_map.get(norm_dua(w).lower())
        if t is None or (w.isupper() and len(w) > 1): return w      # mot inconnu ou sigle (FCFA) : intact
        return t.capitalize() if w[:1].isupper() else t
    parts = re.split(r'(\{[^{}]*\})', text)                      # ne jamais toucher aux {variables}
    return ''.join(p if p.startswith('{') else re.sub(r"[^\W\d_]+(?:['’][^\W\d_]+)*", f, p) for p in parts)
print('mots avec tons restaurables :', len(tone_map))

train_pairs = list(zip(train.fr, train.dua)) + sent_tr * VALIDATED_UPSAMPLE
train_ex = mk(train_pairs); random.shuffle(train_ex)

1 ligne(s) avec colonnes duala/français inversées : remises dans le bon sens
dictionnaire (ok) : 55 phrases de plus de 8 mots CONSERVÉES (traitées comme phrases natives) | 14 écartées (longueurs incompatibles) | 0 de plus de 80 mots
  écartées car longueurs incompatibles (ex. un mot en face de 11 mots) :
                francais                                                        duala
12999          à l’école                   ó esukúlu ɓáɓɔ́ ɓé ndé ó mulemlem má klase
13428          continent         dongo là wâsē (Afrika, Asia, Eropa, India na Osiana)
14604  eyala á muɓɛ́nɛko  eyala á muɓɛ́nɛko é maɓolánɛ́ ó ɓámsɛ na kíka lambo muɓɛ...
17824      homosexualité  mulatáko ó ndóló esiɓɛ ndandó ó momé na momé tɔ múto na ...
18486          un enfant                  « a tǎm eɓóndā » il a touché le pion eɓóndā
18487    les yeux fermés                  « a tǎm eɓóndā » il a touché le pion eɓóndā
18489    quand il réusit                  « a tǎm eɓóndā » il a touché le pion eɓóndā
23587 

## 4. Modèle et vérification des caractères

In [6]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, get_linear_schedule_with_warmup
from transformers.optimization import Adafactor
import transformers; transformers.logging.set_verbosity_error()   # coupe l'avertissement max_length répété
tok = AutoTokenizer.from_pretrained(MODEL, src_lang=SRC, tgt_lang=TGT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, dropout=DROPOUT, attention_dropout=0.1).to(dev)
if FREEZE_EMB:
    for p in model.get_input_embeddings().parameters(): p.requires_grad = False
BOS = tok.convert_tokens_to_ids(TGT)

# caractères inconnus du tokenizer (<unk>) : doit rester < 1 %
sample = train.dua.sample(min(3000, len(train)), random_state=1).tolist()
ids = tok(text_target=sample).input_ids
unk = sum(i.count(tok.unk_token_id) for i in ids) / sum(len(i) for i in ids)
print(f'taux <unk> sur le duala : {unk:.2%}')
if unk > 0.01: print('⚠ trop élevé : repérer les caractères fautifs ci-dessous et les remplacer dans norm_dua()')
print(sorted({c for s in sample for c in s if tok.unk_token_id in tok(text_target=c).input_ids}))
# longueur en tokens des phrases natives : doit rester sous MAX_LEN, sinon elles seraient tronquées à l'entraînement
if sent_tr:
    _m = mk(sent_tr); _L = [len(tok(f).input_ids) for f, _ in _m]; _Ld = [len(tok(text_target=g).input_ids) for _, g in _m]
    print(f'phrases natives (tokens) : source médiane {np.median(_L):.0f}, max {max(_L)} | cible médiane {np.median(_Ld):.0f}, max {max(_Ld)} | MAX_LEN = {MAX_LEN}')
    _cut = np.mean([max(a, b) > MAX_LEN for a, b in zip(_L, _Ld)])
    if _cut > 0: print(f'⚠ {_cut:.1%} des phrases natives seraient tronquées : augmenter MAX_LEN (et réduire BS si la mémoire GPU manque)')


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

taux <unk> sur le duala : 0.01%
['»']
phrases natives (tokens) : source médiane 24, max 44 | cible médiane 29, max 54 | MAX_LEN = 160


## 5. Entraînement (arrêt anticipé = anti sur-apprentissage)
À chaque époque : perte train vs perte validation + chrF de validation. On garde le **meilleur** modèle sur le chrF validation et on s'arrête
si rien ne s'améliore pendant `PATIENCE` époques. Le meilleur modèle est gardé en mémoire (pas d'écriture disque à chaque époque) ; l'entraînement s'arrête après `EPOCHS` époques, `PATIENCE` époques sans progrès, ou `MAX_MINUTES` minutes.

In [7]:
MAX_TOK = 1200    # tokens max par lot (protège la mémoire GPU : le vocabulaire fait 256 000 entrées)
def tokenize(ex):
    S = tok([s for s, _ in ex], max_length=MAX_LEN, truncation=True).input_ids
    T = tok(text_target=[t for _, t in ex], max_length=MAX_LEN, truncation=True).input_ids
    return list(zip(S, T))

def make_batches(tk, bs, shuffle=True):
    idx = list(range(len(tk)))
    if shuffle: random.shuffle(idx)
    L = lambda i: max(len(tk[i][0]), len(tk[i][1]))
    out = []
    for c in range(0, len(idx), bs * 50):                  # lots de longueurs voisines = beaucoup moins de padding
        cur, mx = [], 0
        for i in sorted(idx[c:c + bs * 50], key=L):
            if cur and (len(cur) >= bs or max(mx, L(i)) * (len(cur) + 1) > MAX_TOK):
                out.append(cur); cur, mx = [], 0
            cur.append(i); mx = max(mx, L(i))
        if cur: out.append(cur)
    if shuffle: random.shuffle(out)
    return out

def _pad(seqs, v):
    n = max(map(len, seqs)); return torch.tensor([s + [v] * (n - len(s)) for s in seqs])

def collate(tk, b):
    src = _pad([tk[i][0] for i in b], tok.pad_token_id)
    lab = _pad([tk[i][1] for i in b], -100)
    dec = _pad([tk[i][1] for i in b], tok.pad_token_id)
    ds = model.config.decoder_start_token_id if model.config.decoder_start_token_id is not None else tok.eos_token_id
    dec = torch.cat([torch.full((len(b), 1), ds), dec[:, :-1]], 1)   # décalage à droite, comme HF avec labels=
    return dict(input_ids=src.to(dev), attention_mask=(src != tok.pad_token_id).long().to(dev),
                decoder_input_ids=dec.to(dev), labels=lab.to(dev))

def loss_fn(batch):
    lab = batch['labels']; m = lab != -100
    out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], decoder_input_ids=batch['decoder_input_ids'])
    lp = torch.log_softmax(out.logits[m].float(), -1)       # seulement les positions non vides, et plus de perte HF en double
    nll = -lp.gather(-1, lab[m].unsqueeze(-1)).squeeze(-1)
    smooth = -lp.mean(-1)
    return ((1 - LS) * nll + LS * smooth).mean(), nll.sum().item(), int(m.sum())

@torch.no_grad()
def translate_raw(srcs, beams=4, bs=64):
    model.eval(); order = sorted(range(len(srcs)), key=lambda i: len(srcs[i])); res = [None] * len(srcs)   # longueurs voisines = plus rapide
    for k in range(0, len(order), bs):
        ids_ = order[k:k + bs]
        enc = tok([srcs[i] for i in ids_], return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(dev)
        mx = min(MAX_LEN * 2, int(enc['input_ids'].shape[1] * 2) + 16)          # longueur de sortie proportionnelle à la phrase
        with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
            g = model.generate(**enc, forced_bos_token_id=BOS, num_beams=beams, max_new_tokens=mx, no_repeat_ngram_size=NO_REPEAT)
        for i, o in zip(ids_, tok.batch_decode(g, skip_special_tokens=True)): res[i] = o
    return res

@torch.no_grad()
def translate_scored(srcs, beams=5):
    """Comme translate_raw, mais renvoie aussi le score du modèle (log-probabilité moyenne par token ; plus proche de 0 = plus sûr)."""
    model.eval()
    enc = tok(srcs, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(dev)
    mx = min(MAX_LEN * 2, int(enc['input_ids'].shape[1] * 2) + 16)
    with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
        g = model.generate(**enc, forced_bos_token_id=BOS, num_beams=max(beams, 2), max_new_tokens=mx, no_repeat_ngram_size=NO_REPEAT,
                           return_dict_in_generate=True, output_scores=True)
    return tok.batch_decode(g.sequences, skip_special_tokens=True), g.sequences_scores.float().cpu().tolist()

def eval_sent(pairs, n=100, beams=1):
    """chrF sur des phrases natives (variables restaurées). None s'il n'y en a pas."""
    if not pairs: return None
    pairs = pairs[:n]; src_ = [protect(fr)[0] for fr, _ in pairs]
    hy = [restore(h, protect(fr)[1]) for h, (fr, _) in zip(translate_raw(src_, beams=beams, bs=16), pairs)]
    return round(sacrebleu.corpus_chrf(hy, [[du for _, du in pairs]]).score, 2)

def evaluate(df, n=None, beams=4):
    """Un terme français a souvent plusieurs équivalents duala : on score contre le meilleur.
    n -> sous-échantillon ALÉATOIRE fixe (l'ancien iloc[:n] prenait les mots de A à ~D seulement)."""
    g = df.groupby('key').agg(fr=('fr', 'first'), refs=('dua', list))
    if n and n < len(g): g = g.sample(n, random_state=SEED)
    hyp = translate_raw(g.fr.tolist(), beams=beams)
    items = pd.DataFrame(dict(fr=g.fr.values, hyp=hyp, refs=g.refs.values))
    items['chrf'] = [max(sacrebleu.sentence_chrf(h, [r]).score for r in rs) for h, rs in zip(items.hyp, items.refs)]
    items['exact'] = [float(h.strip().lower() in {r.lower() for r in rs}) for h, rs in zip(items.hyp, items.refs)]
    k = max(len(r) for r in g.refs); R = [[r[min(i, len(r)-1)] for r in g.refs] for i in range(k)]
    m = dict(BLEU=round(sacrebleu.corpus_bleu(hyp, R).score, 2), chrF=round(items.chrf.mean(), 2), exact_pct=round(100 * items.exact.mean(), 2))
    return m, items

val_tok = tokenize(mk(list(zip(val.fr, val.dua))))
def val_loss():
    model.eval(); tot = cnt = 0
    with torch.no_grad():
        for b in make_batches(val_tok, 64, False):
            _, s, c = loss_fn(collate(val_tok, b)); tot += s; cnt += c
    return tot / cnt

train_tok = tokenize(train_ex)
nb = len(make_batches(train_tok, BS)); steps = int(EPOCHS * nb * 1.05)
print(f'{len(train_tok)} exemples | {nb} pas par époque | {EPOCHS} époques max | budget {MAX_MINUTES} min')
opt = Adafactor([p for p in model.parameters() if p.requires_grad], lr=LR, scale_parameter=False, relative_step=False, weight_decay=0.01)
sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
scaler = torch.amp.GradScaler('cuda', enabled=dev == 'cuda')
snap = lambda: {n_: p.detach().cpu().clone() for n_, p in model.named_parameters() if p.requires_grad}   # meilleur modèle gardé en RAM (pas d'écriture disque à chaque époque)
best, bad, hist, best_sd, T0 = -1, 0, [], None, time.time()
for ep in range(1, EPOCHS + 1):
    model.train(); tl = tc = 0; t0 = time.time()
    for step, b in enumerate(make_batches(train_tok, BS), 1):
        batch = collate(train_tok, b)
        with torch.autocast(dev, dtype=torch.float16, enabled=dev == 'cuda'):
            loss, s, c = loss_fn(batch)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sch.step(); tl += s; tc += c
        if step % 200 == 0: print(f'  époque {ep} : {step}/{nb} pas, {(time.time()-t0)/step:.2f} s/pas', flush=True)
    vl = val_loss(); m, _ = evaluate(val, n=EVAL_N, beams=EVAL_BEAMS)
    cs = eval_sent(sent_va) if len(sent_va) >= 10 else None                  # chrF sur phrases natives de validation
    crit = m['chrF'] if cs is None else (m['chrF'] + cs) / 2                 # critère de sélection : dictionnaire ET phrases (si disponibles)
    hist.append(dict(epoch=ep, train_nll=tl / tc, val_nll=vl, **m, chrF_phrases=cs, critere=round(crit, 2)))
    print(hist[-1], f'{time.time()-t0:.0f}s | total {(time.time()-T0)/60:.0f} min', flush=True)
    if crit > best:
        best, bad, best_sd = crit, 0, snap()
        if SAVE_EVERY and ep % SAVE_EVERY == 0: model.save_pretrained(CKPT_DIR); tok.save_pretrained(CKPT_DIR)
    else:
        bad += 1
        if bad >= PATIENCE: print('arrêt anticipé'); break
    if (time.time() - T0) / 60 > MAX_MINUTES: print(f'budget de {MAX_MINUTES} min atteint : arrêt, meilleur modèle conservé'); break
TRAIN_MINUTES = round((time.time() - T0) / 60, 1)
model.load_state_dict(best_sd, strict=False)               # remet le MEILLEUR modèle, puis une seule écriture disque
model.save_pretrained(CKPT_DIR); tok.save_pretrained(CKPT_DIR)
h = pd.DataFrame(hist); print(h.round(3).to_string(index=False))
b_nll, b_chrf = int(h.val_nll.idxmin()) + 1, int(h.critere.idxmax()) + 1
print(f"val_nll minimale : époque {b_nll} | meilleur critère (chrF dictionnaire + phrases) : époque {b_chrf} (modèle retenu) | écart train/val final : {h.val_nll.iloc[-1]-h.train_nll.iloc[-1]:.2f} | durée {TRAIN_MINUTES} min")
print("BLEU : instable sur des termes de 1-2 mots, ne pas s'y fier ; décider sur chrF et exact_pct.")


23227 exemples | 738 pas par époque | 14 époques max | budget 100 min
  époque 1 : 200/738 pas, 0.33 s/pas
  époque 1 : 400/738 pas, 0.33 s/pas
  époque 1 : 600/738 pas, 0.33 s/pas
{'epoch': 1, 'train_nll': 3.9812028708711136, 'val_nll': 3.2544006239906818, 'BLEU': 17.63, 'chrF': np.float64(11.91), 'exact_pct': np.float64(2.8), 'chrF_phrases': None, 'critere': np.float64(11.91)} 249s | total 4 min
  époque 2 : 200/738 pas, 0.33 s/pas
  époque 2 : 400/738 pas, 0.33 s/pas
  époque 2 : 600/738 pas, 0.33 s/pas
{'epoch': 2, 'train_nll': 3.1373120228537417, 'val_nll': 2.934696565449592, 'BLEU': 4.63, 'chrF': np.float64(12.52), 'exact_pct': np.float64(2.8), 'chrF_phrases': None, 'critere': np.float64(12.52)} 249s | total 8 min
  époque 3 : 200/738 pas, 0.33 s/pas
  époque 3 : 400/738 pas, 0.33 s/pas
  époque 3 : 600/738 pas, 0.33 s/pas
{'epoch': 3, 'train_nll': 2.7522488676209367, 'val_nll': 2.77663146747362, 'BLEU': 11.56, 'chrF': np.float64(16.18), 'exact_pct': np.float64(4.8), 'chrF_phrase

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  époque 5 : 200/738 pas, 0.33 s/pas
  époque 5 : 400/738 pas, 0.33 s/pas
  époque 5 : 600/738 pas, 0.33 s/pas
{'epoch': 5, 'train_nll': 2.201555543735755, 'val_nll': 2.621458109606328, 'BLEU': 12.26, 'chrF': np.float64(21.25), 'exact_pct': np.float64(8.8), 'chrF_phrases': None, 'critere': np.float64(21.25)} 248s | total 23 min
  époque 6 : 200/738 pas, 0.33 s/pas
  époque 6 : 400/738 pas, 0.33 s/pas
  époque 6 : 600/738 pas, 0.33 s/pas
{'epoch': 6, 'train_nll': 1.9995018236207223, 'val_nll': 2.553507144140061, 'BLEU': 11.28, 'chrF': np.float64(25.32), 'exact_pct': np.float64(13.0), 'chrF_phrases': None, 'critere': np.float64(25.32)} 247s | total 27 min
  époque 7 : 200/738 pas, 0.33 s/pas
  époque 7 : 400/738 pas, 0.33 s/pas
  époque 7 : 600/738 pas, 0.33 s/pas
{'epoch': 7, 'train_nll': 1.8300664369923483, 'val_nll': 2.527855256150573, 'BLEU': 16.37, 'chrF': np.float64(26.48), 'exact_pct': np.float64(14.2), 'chrF_phrases': None, 'critere': np.float64(26.48)} 249s | total 31 min
  époq

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  époque 9 : 200/738 pas, 0.33 s/pas
  époque 9 : 400/738 pas, 0.33 s/pas
  époque 9 : 600/738 pas, 0.33 s/pas
{'epoch': 9, 'train_nll': 1.5823462711136163, 'val_nll': 2.4968512579021476, 'BLEU': 11.96, 'chrF': np.float64(27.75), 'exact_pct': np.float64(15.0), 'chrF_phrases': None, 'critere': np.float64(27.75)} 249s | total 43 min
  époque 10 : 200/738 pas, 0.33 s/pas
  époque 10 : 400/738 pas, 0.33 s/pas
  époque 10 : 600/738 pas, 0.33 s/pas
{'epoch': 10, 'train_nll': 1.4828449299690867, 'val_nll': 2.5041262939335582, 'BLEU': 13.78, 'chrF': np.float64(28.37), 'exact_pct': np.float64(14.8), 'chrF_phrases': None, 'critere': np.float64(28.37)} 249s | total 47 min
  époque 11 : 200/738 pas, 0.33 s/pas
  époque 11 : 400/738 pas, 0.33 s/pas
  époque 11 : 600/738 pas, 0.33 s/pas
{'epoch': 11, 'train_nll': 1.4032221051114322, 'val_nll': 2.4991986674647175, 'BLEU': 15.86, 'chrF': np.float64(30.0), 'exact_pct': np.float64(17.0), 'chrF_phrases': None, 'critere': np.float64(30.0)} 249s | total 51

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 epoch  train_nll  val_nll  BLEU  chrF  exact_pct chrF_phrases  critere
     1      3.981    3.254 17.63 11.91        2.8         None    11.91
     2      3.137    2.935  4.63 12.52        2.8         None    12.52
     3      2.752    2.777 11.56 16.18        4.8         None    16.18
     4      2.451    2.686  9.95 19.61        7.2         None    19.61
     5      2.202    2.621 12.26 21.25        8.8         None    21.25
     6      2.000    2.554 11.28 25.32       13.0         None    25.32
     7      1.830    2.528 16.37 26.48       14.2         None    26.48
     8      1.696    2.505 14.32 27.19       14.2         None    27.19
     9      1.582    2.497 11.96 27.75       15.0         None    27.75
    10      1.483    2.504 13.78 28.37       14.8         None    28.37
    11      1.403    2.499 15.86 30.00       17.0         None    30.00
    12      1.336    2.518 15.98 29.42       16.2         None    29.42
    13      1.285    2.516 16.27 29.19       16.2         None  

## 6. Évaluation finale (une seule fois, jeux jamais vus)

In [10]:
res, items = evaluate(test); print('TEST dictionnaire (termes jamais vus) :', res)
def ci(x, B=2000):
    rng = np.random.default_rng(0); x = np.asarray(x, float)
    b = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]; return np.percentile(b, [2.5, 97.5]).round(2)
print('IC95 % chrF :', ci(items.chrf), '| IC95 % exact :', ci(100 * items.exact))
copy_chrf = np.mean([max(sacrebleu.sentence_chrf(f, [r]).score for r in rs) for f, rs in zip(items.fr, items.refs)])
print(f'Référence « recopier le français » : chrF {copy_chrf:.2f}  (le modèle doit faire nettement mieux)')
print(items.sample(25, random_state=1)[['fr', 'hyp', 'refs']].to_string(max_colwidth=40))
if sent_te:
    se = mk(sent_te); hy = translate_raw([s for s, _ in se])
    hy = [restore(h, protect(fr)[1]) for h, (fr, _) in zip(hy, sent_te)]
    refs = [du for _, du in sent_te]
    print('TEST SITE (phrases natives) : BLEU', round(sacrebleu.corpus_bleu(hy, [refs]).score, 2),
          '| chrF', round(sacrebleu.corpus_chrf(hy, [refs]).score, 2),
          '| phrases exactes', f'{np.mean([a.lower()==b.lower() for a,b in zip(hy,refs)]):.1%}',
          '| variables OK', f'{np.mean([same_vars(a,f) for a,(f,_) in zip(hy,sent_te)]):.1%}')
else:
    print('Pas de phrases natives validées : aucun BLEU « site » mesurable (il faut des références humaines).')

# résultats de test gardés pour le fichier des meilleurs paramètres (cellule 8)
RES_TEST = dict(dictionnaire=dict(res), ic95_chrf=[float(v) for v in ci(items.chrf)], ic95_exact=[float(v) for v in ci(100 * items.exact)])
if sent_te:
    RES_TEST['site'] = dict(BLEU=round(sacrebleu.corpus_bleu(hy, [refs]).score, 2), chrF=round(sacrebleu.corpus_chrf(hy, [refs]).score, 2),
                            phrases_exactes=round(float(np.mean([a.lower() == b.lower() for a, b in zip(hy, refs)])), 4),
                            variables_ok=round(float(np.mean([same_vars(a, f) for a, (f, _) in zip(hy, sent_te)])), 4))

TEST dictionnaire (termes jamais vus) : {'BLEU': 16.57, 'chrF': np.float64(29.44), 'exact_pct': np.float64(15.64)}
IC95 % chrF : [26.99 31.9 ] | IC95 % exact : [13.01 18.27]
Référence « recopier le français » : chrF 9.95  (le modèle doit faire nettement mieux)
                              fr            hyp                                     refs
135                       en bas          owase                                  [owase]
656       un chant de piroguiers  ŋgeŋge a ɓɔlɔ                               [dingenge]
704                          vue       ɓenyɔedi                                   [jɛnɛ]
392                 le sacrifice         Jjabea                      [Jjabea, musangise]
293               la persécution     betakisedi                                [mibupan]
676  une circoncision incomplète     mukwäñkwäñ                                   [s0k6]
481                        mpétɛ          mpetɛ                            [buka, mpetɛ]
486                 mâchoir

## 7. Traducteur de production : variables + synonymes + glossaire + modèle + garde-fous
Ordre : phrase validée par un natif → (mots inconnus remplacés par un synonyme du dictionnaire) → terme du glossaire (court) → modèle.
Le modèle produit du duala **sans tons** ; `restore_tones` les remet quand la forme tonale est sûre.
Chaque traduction est accompagnée d'indicateurs : variables, synonymes utilisés, mots restés inconnus, phrase découpée, longueur inhabituelle, score du modèle bas. Un indicateur = **à relire par un natif**.


In [11]:
# NB : on construit le glossaire sur TOUT le dictionnaire (le test est déjà mesuré) ; 1 duala par terme = le plus fréquent
gloss = d.groupby('key').duala.agg(lambda s: s.value_counts().index[0]).to_dict()   # forme d'origine, avec tons
native = dict(native_raw)                                                            # forme du locuteur, avec tons
ndua = d.groupby('key').dua.nunique().to_dict()                                      # nombre de traductions duala par terme français (moins = plus sûr)

# ---------- synonymes : mot français absent du vocabulaire -> synonyme dont la traduction duala existe dans le dictionnaire ----------
from functools import lru_cache
LET = re.compile(r"[^\W\d_]+")
STOP = set("""le la les l un une des du de d et ou mais donc or ni car que qui quoi dont où à au aux en dans sur sous par pour avec sans chez vers entre
ce cet cette ces se s sa son ses leur leurs mon ma mes ton ta tes notre nos votre vos je tu il elle on nous vous ils elles me te lui y ne pas plus n m t qu
est sont être avoir a ont suis es sommes êtes été ai as avons avez très aussi comme si cela ça ceci non oui""".split())
def _strip(w): return unicodedata.normalize('NFD', w).encode('ascii', 'ignore').decode().lower()
vocab_fr = {w for t in list(d.fr) + [f for f, _ in sent_tr + sent_va + sent_te] for w in LET.findall(t.lower())}
vocab_n = {_strip(w) for w in vocab_fr}
dico1 = {k for k in gloss if LET.fullmatch(k)}                                      # termes français d'un seul mot, avec une traduction duala
def connu(w): return w in vocab_fr or _strip(w) in vocab_n or (w.endswith(('s', 'x')) and w[:-1] in vocab_fr)

try:
    from nltk.corpus import wordnet as wn
    wn.synsets('maison', lang='fra'); WN_OK = True
except Exception as e:
    WN_OK = False; print('⚠ WordNet français indisponible : synonymes désactivés (', str(e)[:70].replace(chr(10), ' '), ')')

@lru_cache(maxsize=None)
def synonymes_connus(w):
    """Synonymes de w (WordNet) qui ont une traduction duala dans le dictionnaire, du plus sûr au moins sûr."""
    if not WN_OK: return ()
    sc = collections.Counter()
    for rank, s in enumerate(wn.synsets(w, lang='fra')):                              # les sens sont triés du plus courant au plus rare
        for l in s.lemma_names('fra'):
            l = l.replace('_', ' ').lower()
            if l != w and l in dico1: sc[l] += 1 / (rank + 1)
    best = [(l, round(v, 2)) for l, v in sc.items() if v >= MIN_SYN]
    best.sort(key=lambda t: (-t[1], ndua.get(t[0], 99), t[0]))                         # à score égal : le terme le moins ambigu en duala
    return tuple(best[:3])

def substitute(text):
    """-> (texte, [(mot, synonyme)], [mots restés inconnus]). Ne touche ni aux {variables}, ni aux sigles, ni aux noms propres probables."""
    subs, inconnus = [], []
    def f(m):
        w = m.group(0); lw = w.lower()
        if len(lw) <= 3 or lw in STOP or connu(lw): return w
        if w.isupper() or (m.start() > 0 and w[0].isupper()): return w                  # sigle / nom propre probable
        c = synonymes_connus(lw) or (synonymes_connus(lw[:-1]) if lw.endswith(('s', 'x')) else ())
        if not c: inconnus.append(w); return w
        subs.append((w, c[0][0])); return c[0][0].capitalize() if w[0].isupper() else c[0][0]
    parts = re.split(r'(\{[^{}]*\})', text)
    return ''.join(p if p.startswith('{') else LET.sub(f, p) for p in parts), subs, inconnus

# ---------- traduction de phrases longues (découpage si > MAX_LEN tokens) ----------
def translate_long(f, beams=5):
    """f : texte avec variables protégées. -> (duala brut, score minimal, nb de morceaux, tronqué ?)"""
    lim = MAX_LEN - 4; parts = [f]
    if len(tok(f).input_ids) > lim:
        parts, cur = [], ''
        for s_ in re.split(r'(?<=[.!?;:])\s+', f):
            cand = (cur + ' ' + s_).strip()
            if cur and len(tok(cand).input_ids) > lim: parts.append(cur); cur = s_
            else: cur = cand
        if cur: parts.append(cur)
    outs, scs = [], []
    for p in parts:
        t_, s_ = translate_scored([p], beams); outs.append(t_[0]); scs.append(s_[0])
    return ' '.join(outs), min(scs), len(parts), any(len(tok(p).input_ids) > lim for p in parts)

# ---------- calibration des indicateurs sur les phrases natives (jamais inventée : calculée sur vos données) ----------
RATIO, SCORE_MIN = (0.3, 3.0), None
if len(sent_tr) >= 20:
    r_ = np.array([len(du.split()) / max(1, len(fr.split())) for fr, du in sent_tr if len(fr.split()) >= 4])
    if len(r_) >= 20: RATIO = (float(np.percentile(r_, 2)), float(np.percentile(r_, 98)))
print('rapport de longueur duala/français attendu (2e-98e centile des phrases natives) : %.2f - %.2f' % RATIO)
_cal = sent_te if len(sent_te) >= 10 else sent_va if len(sent_va) >= 10 else []
if _cal:
    _src = [protect(fr)[0] for fr, _ in _cal]; _hy, _sc = [], []
    for i in range(0, len(_src), 16):
        h_, s_ = translate_scored(_src[i:i+16]); _hy += h_; _sc += s_
    _ch = [sacrebleu.sentence_chrf(restore(h, protect(fr)[1]), [du]).score for h, (fr, du) in zip(_hy, _cal)]
    rho = float(np.corrcoef(pd.Series(_sc).rank(), pd.Series(_ch).rank())[0, 1])
    print(f'corrélation (rang) score du modèle / chrF réel sur {len(_cal)} phrases natives : {rho:.2f}')
    if rho >= 0.3: SCORE_MIN = float(np.percentile(_sc, 25)); print(f'  -> score informatif : seuil « score bas » = {SCORE_MIN:.2f} (25 % des phrases les moins sûres)')
    else: print('  -> score peu informatif sur vos données : il est affiché mais ne déclenche aucun indicateur')
else:
    print('moins de 10 phrases natives de test/validation : score affiché mais non calibré (aucun indicateur « score bas »)')

def translate(fr, modele_seul=False, synonymes=None, beams=5):
    synonymes = SYNONYMES if synonymes is None else synonymes
    fr = norm_fr(fr); k = fr.lower()
    if not modele_seul and k in native:
        return dict(dua=native[k], brut='', source='natif', ok=True, score=None, subs=[], inconnus=[], flags=[])
    src_, subs, inconnus = substitute(fr) if synonymes else (fr, [], [])
    f, vs = protect(src_)
    if not modele_seul and not vs and len(src_.split()) <= 3 and src_.lower() in gloss:
        return dict(dua=gloss[src_.lower()], brut='', source='glossaire' + (' (via synonyme)' if subs else ''), ok=True, score=None,
                    subs=subs, inconnus=inconnus, flags=['synonyme utilisé'] if subs else [])
    brut_, score, n_parts, tronque = translate_long(f, beams)
    brut = restore(brut_, vs); out = restore(restore_tones(brut_), vs)
    ok = same_vars(fr, out); flags = []
    if not ok: flags.append('variables différentes')
    if subs: flags.append('synonyme utilisé')
    if inconnus: flags.append('mot(s) hors vocabulaire')
    if n_parts > 1: flags.append(f'phrase découpée en {n_parts}')
    if tronque: flags.append('phrase trop longue : tronquée')
    nf = len(fr.split())
    if nf >= 4:
        ratio = len(out.split()) / nf
        if not (RATIO[0] <= ratio <= RATIO[1]): flags.append(f'longueur inhabituelle (x{ratio:.1f})')
    if SCORE_MIN is not None and score < SCORE_MIN: flags.append('score du modèle bas')
    return dict(dua=out, brut=brut, source='modele', ok=ok, score=round(score, 2), subs=subs, inconnus=inconnus, flags=flags)

for s in ['Solde', 'Montant à envoyer', 'Vous allez envoyer {montant} FCFA à {beneficiaire}.', 'Confirmer le transfert']: print(s, '->', translate(s))


rapport de longueur duala/français attendu (2e-98e centile des phrases natives) : 0.74 - 2.30
moins de 10 phrases natives de test/validation : score affiché mais non calibré (aucun indicateur « score bas »)
Solde -> {'dua': 'I yom i yi wè', 'brut': '', 'source': 'glossaire', 'ok': True, 'score': None, 'subs': [], 'inconnus': [], 'flags': []}
Montant à envoyer -> {'dua': 'Libim u lama ñom', 'brut': '', 'source': 'glossaire', 'ok': True, 'score': None, 'subs': [], 'inconnus': [], 'flags': []}
Vous allez envoyer {montant} FCFA à {beneficiaire}. -> {'dua': 'U mɛ́ŋgisɛ FCFA na [1]', 'brut': 'U mɛŋgisɛ FCFA na [1]', 'source': 'modele', 'ok': False, 'score': -1.0, 'subs': [], 'inconnus': ['allez'], 'flags': ['variables différentes', 'mot(s) hors vocabulaire', 'longueur inhabituelle (x0.7)']}
Confirmer le transfert -> {'dua': 'Bamsæ', 'brut': 'Bamsæ', 'source': 'modele', 'ok': True, 'score': -0.36, 'subs': [], 'inconnus': [], 'flags': []}


## 7b. Brouillon pour les traducteurs (post-édition)
Le modèle ne connaît pas le vocabulaire bancaire : ses sorties sont des **propositions à corriger**, pas des traductions. Cette cellule les écrit dans une colonne séparée
« Brouillon modèle » du xlsx (jamais dans « Traduction duala », donc jamais réinjecté à l'entraînement sans validation).

In [12]:
if SENT_XLSX:
    import openpyxl
    wb = openpyxl.load_workbook(SENT_XLSX); ws = wb[sheet]
    col = ws.max_column + 1; ws.cell(1, col).value = 'Brouillon modèle (à corriger)'
    rows = [i for i in range(2, ws.max_row + 1) if ws.cell(i, FR_COL).value]
    for i in rows:
        t = translate(str(ws.cell(i, FR_COL).value)); ws.cell(i, col).value = t['dua'] + (('  [À RELIRE : ' + ', '.join(t['flags']) + ']') if t['flags'] else '')
    ws.column_dimensions[openpyxl.utils.get_column_letter(col)].width = 52
    wb.save('phrases_avec_brouillon.xlsx');
    if files: files.download('phrases_avec_brouillon.xlsx')
    else: print('fichier écrit : phrases_avec_brouillon.xlsx')

## 7c. Tester mes propres phrases
Écrire des phrases françaises, une par ligne (longues acceptées). Pour chaque phrase :
- **Traduction** : sortie du traducteur de production (tons restaurés quand ils sont sûrs) ;
- **Source** : `natif` (phrase déjà validée), `glossaire` (terme court) ou `modele` ;
- **Synonymes** : mots absents du dictionnaire remplacés (`mot -> synonyme`) avant traduction. À contrôler : un synonyme peut changer légèrement le sens ;
- **Mots inconnus** : mots sans synonyme exploitable ; le modèle devine alors, la traduction est moins sûre ;
- **Score** : confiance du modèle (plus proche de 0 = plus sûr) ;
- **À relire** : indicateurs déclenchés. Vide = aucun signal d'alerte automatique, **pas** une garantie de justesse ;
- **Sortie brute** : sortie du modèle sans tons.

`MODELE_SEUL = True` ignore glossaire et phrases natives pour juger le modèle lui-même ; `SYNONYMES = False` (cellule de configuration) coupe les remplacements pour comparer avec/sans.
Seul un locuteur natif peut dire si une traduction est correcte.


In [13]:
MODELE_SEUL = True      # True = ne tester que le modèle ; False = comportement de production (natif > glossaire > modèle)
BEAMS = 5
MES_PHRASES = """
Solde
Montant à envoyer
Vous allez envoyer {montant} FCFA à {beneficiaire}.
Confirmer le transfert
Bonjour, comment allez-vous ?
Entrez le numéro de téléphone du bénéficiaire.
Entrez le nom du bénéficiaire.
Entrez le montant à envoyer.
Choisissez un bénéficiaire dans votre liste.
Ajouter un nouveau bénéficiaire
Enregistrer ce bénéficiaire pour la prochaine fois
Bénéficiaire : {beneficiaire}
Numéro du bénéficiaire : {telephone}
Vérifiez bien le numéro : un envoi à un mauvais numéro ne peut pas être récupéré.
Ce numéro n'est pas enregistré dans votre liste de bénéficiaires.
Le nom du bénéficiaire ne correspond pas au numéro saisi.
Pourquoi envoyez-vous cet argent ? (facultatif)
Motif du transfert : {motif}
Ajoutez une référence pour retrouver ce transfert.
Le bénéficiaire recevra l'argent immédiatement.
Le bénéficiaire recevra l'argent sous {delai}.
Le bénéficiaire recevra un SMS de confirmation.
Vous pouvez envoyer de l'argent à un numéro de téléphone camerounais.
Le numéro doit contenir 9 chiffres.
Saisissez le numéro sans espace.
Montant en FCFA
Combien voulez-vous envoyer ?
Votre transfert de {montant} FCFA vers {beneficiaire} a été effectué avec succès.
Transfert réussi
Transaction n° {reference}
Conservez ce numéro de transaction.
Un reçu a été envoyé par SMS.
Télécharger le reçu
Partager le reçu
Nouveau solde : {solde} FCFA
Votre transfert est en cours de traitement.
Votre transfert est en attente de validation.
Votre transaction a échoué. Aucun montant n'a été retiré de votre compte.
Votre transfert a été annulé. Votre argent vous a été rendu.
Le montant sera remboursé sous {delai}.
Merci d'avoir utilisé nos services.
Faire un autre transfert
Retour à l'accueil
Voir mes transactions
Vous avez reçu {montant} FCFA de {expediteur}.
{expediteur} vous a envoyé {montant} FCFA.
Opération terminée le {date} à {heure}.
Votre paiement de {montant} FCFA à {commercant} a été accepté.
Le reçu est disponible dans votre historique.
Solde insuffisant pour effectuer cette opération.
Le numéro saisi est invalide. Vérifiez et réessayez.
Le montant saisi est invalide.
Le montant doit être supérieur à {min} FCFA.
Le montant dépasse votre limite journalière de {limite} FCFA.
Le montant dépasse votre solde disponible.
Votre session a expiré. Veuillez vous reconnecter.
Le service est temporairement indisponible. Réessayez plus tard.
Connexion perdue. Vérifiez votre réseau.
Une erreur est survenue. Veuillez réessayer.
Cette opération n'est pas autorisée.
Le champ « {champ} » est obligatoire.
Le bénéficiaire est introuvable.
Le compte du bénéficiaire est bloqué.
Votre compte est bloqué. Contactez votre banque.
Le code de confirmation a expiré. Demandez un nouveau code.
Trop de tentatives incorrectes. Réessayez dans {minutes} minutes.
Opération refusée.
Le transfert n'a pas pu être effectué. Aucun montant n'a été retiré.
Le délai de traitement est dépassé.
Les informations saisies ne correspondent pas.
Aucun résultat trouvé.
Cette fonction n'est pas disponible pour le moment.
Mise à jour de l'application requise.
Les frais de cette opération sont de {frais} FCFA.
Aucun frais pour cette opération.
Les frais dépendent du montant envoyé.
Pour un montant de {montant} FCFA, les frais sont de {frais} FCFA.
Les frais sont retirés de votre solde en plus du montant envoyé.
Les frais sont inclus dans le montant.
Les frais de service sont de {pourcentage} % du montant.
Frais de transfert : {frais} FCFA
Une taxe de {taxe} FCFA est ajoutée à cette opération.
Le transfert entre comptes de la même banque est gratuit.
Les retraits sont soumis à des frais de {frais} FCFA.
Consultez la grille des frais avant d'envoyer.
Les frais peuvent changer sans préavis.
Montant minimum : {min} FCFA. Montant maximum : {max} FCFA.
Le montant minimum d'un transfert est de {min} FCFA.
Le montant maximum par jour est de {max} FCFA.
Calcul des frais en cours.
Impossible de calculer les frais pour le moment.
Total à payer : {total} FCFA
Vous paierez {frais} FCFA de frais pour envoyer {montant} FCFA.
Environnement de test — données fictives, aucune transaction réelle.
Protocole de test
Retour au hackathon
Banque en ligne
Solde
Frais
Bénéficiaire
Récapitulatif
Confirmation
Frais de transfert
Choisissez le type de transfert et le montant. Les frais sont calculés automatiquement.
Type d'opération
Compte Safinum
Transfert instantané et gratuit
Montant à envoyer
Gratuit
Total à débiter
Solde après opération
Retour
Continuer
Votre compte
Compte courant
Solde disponible
Sécurité
Vérifiez toujours le nom et le numéro du bénéficiaire avant de confirmer un transfert.
Connexion sécurisée
Safinum Pay — environnement de démonstration. Ne saisissez aucune donnée personnelle réelle.
Ne saisissez aucune donnée personnelle réelle.
Renseignez les informations du destinataire. Vérifiez toujours le numéro avant de valider.
Votre solde est de 250 000 FCFA.
Paiement marchand : 12 500 FCFA.
Les frais sont de 150 FCFA.
Vous envoyez 10 000 FCFA.
Total : 10 150 FCFA.
Le montant minimum est de 100 FCFA.
Le montant maximum est de 500 000 FCFA.
Le montant maximum par jour est de 1 000 000 FCFA.
Les frais sont de 1,5 % du montant.
Les frais de 2 % s'ajoutent à chaque retrait.
Il vous reste 75 000 FCFA après cette opération.
Opération du 5 novembre 2026.
Date : lundi 9 novembre 2026.
Il est 14 h 30.
Il vous reste 3 essais.
Le code est valable 5 minutes.
Votre transfert arrivera dans 10 minutes.
Vous avez 2 nouvelles notifications.
Tous les 30 jours.
Vérifiez les informations avant de confirmer.
Vous allez envoyer {montant} FCFA à {beneficiaire}.
Expéditeur : {prenom}
Destinataire : {beneficiaire}
Montant envoyé : {montant} FCFA
Frais : {frais} FCFA
Montant total retiré de votre compte : {total} FCFA
Le montant total retiré de votre compte sera de {total} FCFA (montant : {montant} FCFA + frais : {frais} FCFA).
Votre solde après l'opération sera de {solde} FCFA.
Cette opération ne peut plus être annulée une fois confirmée.
Appuyez sur Confirmer pour valider le transfert, ou sur Annuler pour revenir en arrière.
Modifier les informations
Tout est correct ?
Relisez bien chaque information avant de continuer.
Ce transfert sera traité sous {delai}.
Date de l'opération : {date}
En confirmant, vous acceptez les conditions du service.
Voir les conditions
Si une information est fausse, appuyez sur Modifier.
Vous ne pourrez pas récupérer cet argent en cas d'erreur de saisie.
Cette opération dépasse votre habitude. Vérifiez-la avec soin.
Confirmer le transfert
Bonjour, {prenom}.
Bienvenue sur votre espace bancaire.
Consultez votre solde et lancez un transfert en toute sécurité.
Votre solde disponible est de {montant} FCFA.
Votre solde actuel est de {montant} FCFA.
Solde mis à jour le {date} à {heure}.
Voici vos dernières transactions.
Vous n'avez aucune transaction récente.
Votre compte courant est actif.
Numéro de compte : {numero}
Voulez-vous voir votre solde ?
Votre solde est masqué. Appuyez pour l'afficher.
Appuyez ici pour masquer votre solde.
Vous avez {n} nouvelle notification.
Choisissez une opération.
Que voulez-vous faire aujourd'hui ?
Paiement marchand de {montant} FCFA le {date}.
Dépôt de {montant} FCFA reçu le {date}.
Retrait de {montant} FCFA effectué le {date}.
Compte courant · {numero}
Ne donnez jamais votre code PIN à personne, même pas à un agent de la banque.
Attention : si on vous demande de l'argent pour un gain ou un service, c'est peut-être une arnaque.
Un code de confirmation a été envoyé au {telephone}.
Entrez le code de confirmation reçu par SMS.
Le code est valable {minutes} minutes.
Le code saisi est incorrect. Il vous reste {n} essais.
Ne cliquez pas sur les liens reçus par SMS de numéros inconnus.
Cette opération semble inhabituelle. Confirmez-vous que c'est bien vous ?
Vos informations sont protégées.
Votre compte a été bloqué temporairement pour votre sécurité.
Notre équipe ne vous demandera jamais votre code PIN.
Si vous n'êtes pas à l'origine de cette opération, appelez immédiatement {service_client}.
Choisissez un code PIN difficile à deviner.
Ne partagez votre mot de passe avec personne.
Changez votre code PIN régulièrement.
Déconnectez-vous après avoir utilisé l'application.
Méfiez-vous des personnes qui vous pressent d'envoyer de l'argent.
Une personne inconnue vous demande de l'argent ? Ne répondez pas.
Vous avez reçu un message suspect ? Ne le transférez pas et ne répondez pas.
Une connexion inhabituelle a été détectée sur votre compte.
Votre session va expirer dans {minutes} minutes.
Pour votre sécurité, vous avez été déconnecté.
Saisissez votre code PIN pour confirmer.
Votre code PIN est composé de {n} chiffres.
Personne d'autre ne doit connaître votre code PIN.
Vérifiez que l'adresse du site est correcte avant de saisir vos informations.
L'application ne vous demandera jamais votre code par appel téléphonique.
Appelez le service client en cas de doute.
Vous avez envoyé {montant} FCFA à {beneficiaire}. Frais : {frais} FCFA. Nouveau solde : {solde} FCFA.
Vous avez reçu {montant} FCFA de {expediteur}. Nouveau solde : {solde} FCFA.
Dépôt de {montant} FCFA effectué avec succès.
Retrait de {montant} FCFA effectué chez l'agent {agent}.
Votre code de retrait est {code}. Valable {minutes} minutes.
Paiement de facture réussi : {montant} FCFA.
Achat de crédit de communication : {montant} FCFA.
Votre forfait a été activé.
Facture d'électricité : {montant} FCFA à payer avant le {date}.
Facture d'eau : {montant} FCFA à payer avant le {date}.
Abonnement TV renouvelé pour {montant} FCFA.
Frais de scolarité : {montant} FCFA payés avec succès.
Paiement chez le marchand {commercant} : {montant} FCFA.
Entrez le numéro de l'agent.
Trouvez un agent près de vous.
Rechargez votre compte mobile money.
Transfert vers un autre réseau : frais de {frais} FCFA.
Envoyer de l'argent à l'étranger
Recevoir de l'argent de l'étranger
Taux de change : 1 EUR = {taux} FCFA.
Vous allez recevoir {montant} FCFA dans {delai}.
Votre compte mobile money est activé.
Votre numéro est déjà associé à un compte.
Annuler la dernière transaction
Demander un remboursement
Ouvrez un compte épargne en quelques minutes.
Votre épargne actuelle est de {montant} FCFA.
Le taux d'intérêt annuel est de {taux} %.
Vous gagnerez {interet} FCFA d'intérêts en {mois} mois.
Déposez de l'argent sur votre compte épargne.
Vous pouvez retirer votre épargne à tout moment.
Un retrait avant {date} entraîne des pénalités.
Demandez un crédit de {montant} FCFA.
Durée du crédit : {mois} mois.
Votre mensualité sera de {mensualite} FCFA.
Le montant total à rembourser est de {total} FCFA.
Le taux d'intérêt de ce crédit est de {taux} % par an.
Votre demande de crédit est en cours d'examen.
Votre demande de crédit a été acceptée.
Votre demande de crédit a été refusée.
Prochain remboursement : {montant} FCFA le {date}.
Vous avez un retard de paiement de {n} jours.
Des pénalités de {montant} FCFA s'appliquent en cas de retard.
Remboursez votre crédit avant la date limite.
Garant : personne qui s'engage à rembourser si vous ne pouvez pas.
Vous devez présenter une pièce d'identité.
Votre cotisation de {montant} FCFA est due le {date}.
Tontine : prochain tour de {beneficiaire} le {date}.
Montant de la cotisation : {montant} FCFA par {periode}.
Lisez bien les conditions avant de signer.
Modifier mon numéro de téléphone
Changer mon code PIN
Mot de passe oublié ?
Créer un compte
Bienvenue ! Choisissez votre langue.
Choisissez votre langue préférée.
Changer la langue de l'application
Paramètres du compte
Notifications
Activer les notifications
Voir mon relevé de compte
Télécharger mon relevé
Relevé du {date_debut} au {date_fin}
Mes informations personnelles
Contacter le service client
Le service client est disponible de {heure_debut} à {heure_fin}.
Besoin d'aide ?
Questions fréquentes
Comment envoyer de l'argent ?
Comment vérifier mon solde ?
Comment changer mon code PIN ?
À propos de l'application
Conditions d'utilisation
Politique de confidentialité
Merci de patienter, chargement en cours.
Présentez-vous dans une agence ou chez un agent agréé avec votre carte nationale d'identité en cours de validité.
Si vous perdez votre téléphone, appelez immédiatement le service client pour faire bloquer votre compte. Votre argent restera disponible sur votre nouvelle carte SIM.
Opposition sur carte
Bloquer ma carte
Faites bloquer immédiatement votre carte SIM auprès de votre opérateur et votre compte auprès de votre banque.
Raccrochez et rappelez le numéro officiel inscrit au dos de votre carte ou sur le site de la banque.
Votre carte a été bloquée à votre demande.
Ne prêtez jamais votre compte Mobile Money ni votre carte SIM à un tiers.
Carte bancaire
Carte prépayée
Carte Visa
Carte GIMAC
Carte capturée
Mes cartes
Gérer ma carte
Recharger ma carte prépayée
Bloquer temporairement la carte
Débloquer la carte
Commander une nouvelle carte
Numéro de carte (16 chiffres)
Nom inscrit sur la carte
Que faire si le distributeur a avalé ma carte ?
Puis-je utiliser ma carte à l'étranger ?
Combien de temps faut-il pour recevoir une nouvelle carte ?
Comment payer en ligne avec ma carte ?
Oui, si votre carte est internationale et que les paiements à l'étranger sont activés.
Signez votre carte au dos dès sa réception.
Retirez votre carte avant de prendre vos billets.
Vérifiez que rien n'est fixé sur la fente du distributeur avant d'y insérer votre carte.
Insérez votre carte
Paiement par carte de {montant} FCFA chez {nom} le {date}.
Votre nouvelle carte est disponible à l'agence {agence}.
Votre carte expire le {date}. Pensez à la renouveler.
Votre carte prépayée a été rechargée de {montant} FCFA.
Carte expirée.
Carte non autorisée pour les paiements en ligne.
Votre carte sera capturée après trois codes erronés.
Ne communiquez jamais le cryptogramme de votre carte par téléphone.
La carte reste la propriété de la banque et doit être restituée sur simple demande.
La cotisation annuelle de la carte est prélevée à la date anniversaire du contrat.
Le paiement sans contact vous permet de régler vos petits achats en approchant simplement votre carte du terminal.
La carte prépayée ne nécessite pas de compte bancaire : vous la rechargez et vous ne dépensez que l'argent disponible.
Gestion de la carte
Je n'ai pas de carte d'identité, seulement un récépissé. Puis-je ouvrir un compte ?
Si votre téléphone perd soudainement le réseau sans raison, contactez votre opérateur : quelqu'un a peut-être demandé un duplicata de votre carte SIM.
Ne publiez jamais de photo de votre carte bancaire, de votre pièce d'identité ou d'un reçu de transaction sur les réseaux sociaux.
Votre carte a été utilisée pour un paiement à l'étranger de {montant} {devise}. Si vous ne reconnaissez pas cette opération, faites opposition.
L'opposition sur votre carte se terminant par {numero} a été enregistrée le {date} à {heure}.
Le duplicata frauduleux de carte SIM permet à un escroc de recevoir vos codes de validation. Si votre ligne coupe brusquement, réagissez sans attendre.
Le solde disponible tient compte des opérations en cours, comme un paiement par carte pas encore débité ou un chèque en cours d'encaissement.
Les frais mensuels comprennent généralement la tenue de compte, la carte bancaire et les services d'alerte. Le détail figure sur votre relevé.
Le compte salaire est souvent assorti d'avantages, comme une carte gratuite ou une avance sur salaire.
Le paiement en ligne par Mobile Money ne nécessite pas de carte bancaire : une simple validation sur votre téléphone suffit.
Ma carte est bloquée après trois codes faux. Comment la débloquer ?
Ma carte a été refusée chez le commerçant alors que j'ai de l'argent sur mon compte. Pourquoi ?
Puis-je payer sur internet avec ma carte GIMAC ?
J'ai perdu ma carte au marché. Que dois-je faire tout de suite ?
Combien coûte la carte chaque année ?
Contactez votre agence : après vérification de votre identité, un nouveau code peut vous être envoyé ou votre carte débloquée.
Le refus peut venir d'un plafond de paiement atteint, d'une carte expirée ou d'un terminal mal connecté.
Faites opposition immédiatement par l'application ou par téléphone, puis demandez le remplacement de votre carte en agence.
Votre carte a été retenue par le distributeur de l'agence {agence}. Vous pouvez la récupérer sous {duree} avec votre pièce d'identité.
Votre carte se terminant par {numero} est activée. Vous pouvez l'utiliser dès maintenant.
Montant à payer : {montant} FCFA. Insérez ou approchez votre carte.
Opération impossible : votre carte n'est pas acceptée par ce distributeur.
Délai dépassé. Votre carte va vous être restituée.
Carte illisible. Veuillez réessayer ou utiliser une autre carte.
Votre carte est en opposition. Elle ne peut plus être utilisée.
Prenez votre carte, sinon elle sera retenue dans {nombre} secondes.
Si le distributeur vous semble modifié ou si le clavier bouge, n'insérez pas votre carte et prévenez la banque.
Ne quittez jamais votre carte des yeux lorsque vous payez chez un commerçant.
Une carte internationale peut être utilisée à l'étranger. Les paiements en devises sont convertis en FCFA et peuvent entraîner des frais.
Le titulaire s'engage à utiliser la carte dans la limite du solde disponible de son compte.
La carte est strictement personnelle et ne peut être prêtée à un tiers.
La validité de la carte est de {duree} à compter de sa date d'émission.
Comparez nos cartes
Prenez une photo du recto puis du verso de votre carte d'identité, sans reflet et sans couper les bords.
L'application vous permet de bloquer votre carte, de suivre vos dépenses et de recevoir une alerte à chaque opération, sans vous déplacer en agence.
Puis-je payer en euros avec ma carte pendant un voyage en France ?
Oui, si votre carte est internationale. Le montant sera converti en FCFA au cours du jour, avec des frais éventuels.
Paiement par carte
Numéro de compte : {numero_compte}
Mobile Money
Compte Mobile Money
Mon compte
5. Mon compte
Comment ouvrir un compte Mobile Money ?
Quels documents faut-il pour ouvrir un compte Mobile Money ?
Puis-je transférer de l'argent de mon compte bancaire vers Mobile Money ?
Comment recevoir sur mon compte Mobile Money de l'argent envoyé depuis l'étranger ?
Composez {code} pour accéder au menu Mobile Money.
Comptez votre argent devant l'agent avant de partir.
Rappel : votre compte n'a enregistré aucune opération depuis {duree}.
Ce numéro n'est pas enregistré sur Mobile Money.
Opération refusée : votre compte n'est pas encore vérifié.
Nous n'avons pas pu vérifier votre identité. Rendez-vous en agence.
Un compte resté inactif pendant plus de {duree} peut être suspendu.
L'ouverture d'un compte est réservée aux personnes disposant d'une pièce d'identité valide.
Tarifs Mobile Money
Compte à débiter
Provision insuffisante sur le compte à débiter.
Compte débité : {numero_compte}
Blocage du compte
Bloquer mon compte
Quelqu'un tente peut-être d'accéder à votre compte. Ne communiquez ce code à personne et prévenez votre banque.
Faites opposition sans attendre et déposez une réclamation en agence ou auprès du service client.
Conservez le numéro du service client pour pouvoir bloquer votre compte rapidement.
Une connexion à votre compte a été détectée depuis un nouvel appareil le {date} à {heure}. Si ce n'est pas vous, bloquez votre compte immédiatement.
N'installez pas d'application à la demande d'un inconnu qui prétend vous aider.
Si vous recevez un code que vous n'avez pas demandé, quelqu'un essaie probablement d'utiliser votre compte. Ne le transmettez jamais.
Compte joint
Compte professionnel
Compte salaire
Titulaire du compte
Clôture de compte
Numéro de compte
Mes comptes
Ouvrir un compte
Ouvrir un compte en ligne
Type de compte souhaité
Comment ouvrir un compte bancaire ?
Quels documents faut-il pour ouvrir un compte ?
Combien coûte la tenue de compte ?
Puis-je ouvrir un compte sans me déplacer ?
Comment clôturer mon compte ?
Un mineur peut-il avoir un compte ?
Comment ajouter un mandataire sur mon compte ?
Votre RIB est disponible dans l'application, rubrique « Mes comptes », ou en agence.
Adressez une demande écrite et signée à votre agence après avoir régularisé toutes les opérations en cours.
Oui, un compte peut être ouvert au nom d'un mineur par son représentant légal.
Votre compte a été débité de {montant} FCFA le {date}. Référence : {reference}.
Votre salaire de {montant} FCFA a été versé sur votre compte.
Félicitations, votre compte n° {numero_compte} est désormais actif.
Votre compte est à découvert de {montant} FCFA. Merci de l'approvisionner.
Votre chéquier est disponible à l'agence {agence}.
Ce compte est clôturé.
Ce compte est bloqué. Contactez votre agence.
Ne signez jamais un document d'ouverture de compte sans l'avoir lu.
Un compte joint fonctionne sous la signature de l'un ou l'autre des titulaires, sauf mention contraire.
La banque peut clôturer un compte resté sans mouvement pendant plus de {duree}, après en avoir informé le titulaire.
Ouvrez votre compte en quelques minutes depuis votre téléphone, puis finalisez la vérification de votre identité en agence ou par appel vidéo.
Vue d'ensemble de mes comptes
Détail du compte
Ouverture de compte
Profil
Langue
Aide
Changer la langue
Comment changer la langue de l'application ?
Ouvrez « Paramètres », puis « Langue », et choisissez la langue souhaitée.
Rendez-vous en agence ou dans l'application pour souscrire au service SMS banking.
Centre d'aide
Choisissez votre langue
Que faire si le distributeur ne m'a pas donné l'argent mais a débité mon compte ?
N'acceptez pas l'aide d'un inconnu au distributeur.
Compte à terme
Service client
Assistance
Trouver une agence
Appeler le service client
Quels sont les horaires du service client ?
Votre réclamation est traitée dans un délai de {duree} à compter de sa réception.
Notre service client est disponible du lundi au samedi, de {heure} à {heure}, par téléphone, par chat et en agence.
Comment pouvons-nous vous aider ?
Un budget est un plan qui compare ce que vous gagnez et ce que vous dépensez. Il vous aide à éviter les fins de mois difficiles.
Garder son argent sur un compte ou un portefeuille mobile est plus sûr que de le conserver en espèces à la maison.
Présentez-vous en agence avec votre pièce d'identité et votre titre de voyage ; le montant des devises est soumis à la réglementation des changes.
Est-ce que mes enfants peuvent avoir leur propre compte Mobile Money ?
Je change de numéro de téléphone : comment récupérer l'argent de mon ancien compte ?
Mon compte affiche « suspendu ». Qu'est-ce que cela signifie ?
L'ouverture d'un compte nécessite une pièce d'identité valide. Renseignez-vous en agence sur les documents acceptés.
Un agent agréé affiche le logo officiel de l'opérateur et possède un numéro d'agent que vous pouvez vérifier auprès du service client.
L'agent {nom} a déposé {montant} FCFA sur votre compte le {date} à {heure}.
Votre opération n'a pas pu être finalisée. Si votre compte a été débité, le montant vous sera remboursé sous {duree}.
Ne composez jamais un code envoyé par un inconnu : il pourrait servir à vider votre compte.
Si vous vous êtes trompé de numéro, n'attendez pas : appelez le service client dans les plus brefs délais.
Si l'opération échoue, aucun montant n'est débité. Si vous constatez un débit sans confirmation, contactez le service client avec la date et l'heure de l'opération.
Vous pouvez relier votre compte Mobile Money à votre compte bancaire pour transférer de l'argent de l'un à l'autre en quelques secondes.
Relier mon compte bancaire
Vous allez transférer {montant} FCFA du compte {numero_compte} vers le compte de {nom}.
Après validation, les fonds quitteront immédiatement votre compte et ne pourront plus être récupérés par la banque.
Le RIB comprend le code banque, le code guichet, le numéro de compte et la clé RIB. Communiquez-le à votre employeur pour recevoir votre salaire.
Ne confiez jamais votre téléphone déverrouillé à quelqu'un qui propose de « régler votre problème » de compte.
Plusieurs tentatives de connexion échouées ont été détectées sur votre compte. Par précaution, votre accès est bloqué pendant {duree}.
Votre numéro de téléphone associé au compte a été modifié le {date}. Si vous n'avez pas fait cette demande, appelez-nous immédiatement.
Votre adresse e-mail a été modifiée. Si ce n'est pas vous, contactez le service client au plus vite.
Votre compte a été débloqué après vérification. Vous pouvez de nouveau effectuer vos opérations.
Quelqu'un m'a appelé en connaissant mon nom et mon numéro de compte. Comment a-t-il eu ces informations ?
Activez les alertes SMS pour être prévenu de chaque opération sur votre compte.
Votre compte a été verrouillé après plusieurs tentatives infructueuses. Contactez le service client pour le débloquer.
Mon salaire a été viré, mais je ne le vois pas encore sur mon compte. Est-ce normal ?
Puis-je ouvrir un compte pour mon association ?
Que se passe-t-il si mon compte reste longtemps sans mouvement ?
Comment changer l'adresse enregistrée sur mon compte ?
Oui, une association peut ouvrir un compte en présentant ses statuts, le récépissé de déclaration et la liste de ses dirigeants.
Un compte sans mouvement pendant longtemps peut être déclaré inactif. Il suffit en général d'une opération pour le réactiver.
Rien ne vous empêche d'avoir des comptes dans plusieurs établissements, à condition de respecter les règles de chacun.
Votre compte n'a enregistré aucune opération depuis {duree}. Effectuez une opération pour éviter sa mise en sommeil.
Vous n'êtes pas autorisé à consulter ce compte.
Un compte existe déjà avec ce numéro de pièce d'identité.
Sur un compte joint, chaque titulaire est responsable de la totalité des dettes du compte.
Le compte professionnel sépare l'argent de votre activité de vos dépenses personnelles. Il facilite la tenue de vos comptes et vos relations avec vos fournisseurs.
Vous pouvez donner une procuration à une personne de confiance pour qu'elle effectue des opérations sur votre compte en votre absence.
Choisissez le compte qui vous convient
La brochure tarifaire est disponible en agence et sur le site de la banque ; elle doit être affichée de façon visible.
Les tarifs indiqués sont valables à compter du {date} et remplacent les précédents.
Puis-je acheter des unités de compteur prépayé avec Mobile Money ?
Oui : choisissez le fournisseur, saisissez le numéro du compteur et le montant, et vous recevrez le code de recharge par SMS.
Achat d'électricité prépayée de {montant} FCFA réussi. Code de recharge : {code}. Compteur n° {numero}.
Le compteur n° {numero} est introuvable. Vérifiez le numéro et réessayez.
Conservez le code de recharge jusqu'à ce qu'il soit bien saisi sur votre compteur.
Puis-je utiliser l'application dans une autre langue que le français ?
Votre profil a été mis à jour avec succès.
Impossible de charger vos comptes. Tirez vers le bas pour réessayer.
Vous devez avoir au moins 18 ans pour créer un compte en ligne.
Vous pouvez choisir la langue d'affichage de l'application dans les paramètres. Les montants et les références restent affichés tels quels.
Un remboursement de {montant} FCFA a été prélevé sur votre compte. Reste à rembourser : {montant} FCFA.
Décrivez précisément votre projet et la manière dont vous comptez rembourser.
Vous devez passer par une société de bourse agréée, qui ouvre un compte-titres à votre nom et transmet vos ordres d'achat.
Un dividende de {montant} FCFA a été versé sur votre compte au titre des actions {nom}.
Votre indemnisation de {montant} FCFA pour le sinistre {reference} a été versée sur votre compte.
La micro-assurance offre une protection simple et peu coûteuse, souvent payable chaque mois par Mobile Money.
Puis-je faire ma réclamation par téléphone ou dois-je venir en agence ?
Pour mieux vous aider, pouvez-vous nous indiquer la date, l'heure et le montant de l'opération concernée ?
Si la réponse de votre agence ne vous satisfait pas, vous pouvez adresser votre réclamation au service qualité, puis au médiateur.
Copier le numéro de compte
Numéro de compte copié.
Recharger mon compte
Vers un compte Mobile Money
Disponible uniquement pour les comptes vérifiés
Ce montant est inhabituel pour votre compte. Une vérification supplémentaire pourra vous être demandée.
Aide familiale
Ce numéro n'est associé à aucun compte actif.
Numéro vérifié : compte actif.
Compte débité
Le montant total sera débité immédiatement de votre compte.
Action requise : confirmez votre numéro de téléphone pour sécuriser votre compte.
Ce n'est pas moi, bloquer mon compte
Changer de langue
Profil de l'utilisateur
Afficher l'aide
Mon profil
Aide et contact
Service client disponible 24 h/24 et 7 j/7.
Agent Mobile Money
Compte suspendu
Langue de l'application
Rechargez votre compte puis réessayez.
Le numéro saisi n'est pas un numéro Mobile Money valide.
Votre compte est bloqué après trois codes erronés.
Votre session a expiré pour protéger votre compte.
Votre compte n'est pas encore vérifié.
Où trouver un agent Mobile Money près de chez moi ?
Appelez immédiatement le service client pour bloquer votre compte.
Votre compte a été bloqué à votre demande.
Votre compte a été réactivé.
Le compte bancaire {numero_compte} a été relié à votre portefeuille.
Votre demande a bien été prise en compte.
Nom du titulaire du compte
Votre compte a été temporairement bloqué après plusieurs codes erronés. Rendez-vous chez un agent avec votre pièce d'identité pour le débloquer.
N'installez jamais une application de contrôle à distance à la demande d'une personne qui se présente comme agent du service client.
Si un faux agent vous appelle pour obtenir votre code de validation, raccrochez immédiatement. Le service client ne demande jamais ce code.
Vous avez saisi un code de validation incorrect à trois reprises. L'opération a été annulée pour protéger votre compte.
Votre portefeuille peut être relié à votre compte bancaire. Vous pourrez alors transférer de l'argent dans les deux sens sans vous déplacer.
Une erreur technique est survenue pendant le traitement. Si votre compte a été débité, le remboursement sera automatique sous 24 heures.
Votre réclamation {reference} a été traitée. Le montant de {montant} FCFA a été remboursé sur votre compte.
Nous avons détecté une connexion à votre compte depuis un nouveau téléphone. Si ce n'est pas vous, bloquez immédiatement votre compte.
Les mineurs ne peuvent ouvrir un compte Mobile Money qu'avec l'accord d'un parent ou d'un tuteur légal.
Recharge de crédit téléphonique
Acheter du crédit
3. Achat de crédit ou de forfait
Vous avez acheté {montant} FCFA de crédit de communication pour le {numero_telephone}.
N'envoyez jamais d'argent à une personne que vous n'avez jamais rencontrée et qui vous promet un emploi, un prêt ou un héritage.
Protégez votre téléphone comme votre portefeuille : verrouillez-le, ne le prêtez pas et signalez immédiatement sa perte.
Compte d'épargne
Opération au crédit
Votre compte a été crédité de {montant} FCFA le {date}.
Bienvenue {nom} ! Votre espace client est prêt.
Déposez une réclamation avec le ticket du distributeur ; le montant sera recrédité après vérification.
Crédit à la consommation
Prêt immobilier
Prêt scolaire
Crédit de trésorerie
Mensualité
Simulation de crédit
Crédits
Simuler un crédit
Mes prêts en cours
Demander un prêt
Faire une demande de prêt
Objet du prêt
Quels documents faut-il pour demander un prêt ?
Puis-je rembourser mon prêt avant la fin ?
Que se passe-t-il si je ne paie pas une mensualité ?
Le montant dépend de vos revenus et de vos charges : vos mensualités ne doivent pas dépasser une part de votre revenu fixée par la banque.
Lisez attentivement l'offre de prêt avant de la signer.
Vérifiez que la mensualité reste supportable même en cas de dépense imprévue.
Bonne nouvelle : votre prêt de {montant} FCFA a été accordé.
Votre demande de prêt n'a pas pu être acceptée. Votre conseiller peut vous en expliquer les raisons.
Les fonds de votre prêt ont été versés sur votre compte.
Rappel : votre mensualité de {montant} FCFA sera prélevée le {date}.
Votre mensualité du {date} est impayée. Merci d'approvisionner votre compte.
Félicitations, votre prêt est entièrement remboursé.
La durée choisie n'est pas disponible pour ce type de prêt.
Vous avez déjà une demande de prêt en cours.
Un crédit vous engage et doit être remboursé. Vérifiez vos capacités de remboursement avant de vous engager.
Les mensualités sont prélevées automatiquement sur le compte du client à chaque échéance.
Le prêt n'est débloqué qu'après la signature du contrat et la constitution des garanties.
Le crédit à la consommation finance vos projets personnels : équipement de la maison, véhicule, études ou événements familiaux.
Simulez votre crédit en ligne pour connaître le montant de vos mensualités avant de faire votre demande.
Mensualité : {montant} FCFA
Coût total du crédit : {montant} FCFA
Épargne
Plan d'épargne
Épargne programmée
Livret d'épargne
Épargne logement
Mon épargne
Ouvrir un compte d'épargne
Épargner maintenant
Programmer une épargne
Montant à épargner chaque mois
Objectif d'épargne
Quel est le montant minimum pour ouvrir un compte d'épargne ?
Comment épargner automatiquement chaque mois ?
Activez l'épargne programmée : un montant fixe est transféré chaque mois de votre compte courant vers votre épargne.
Votre épargne programmée de {montant} FCFA a été effectuée.
Épargner régulièrement, même de petites sommes, vous aide à faire face aux imprévus et à financer vos projets.
Mes objectifs d'épargne
Votre réclamation a été traitée. Le montant de {montant} FCFA a été recrédité sur votre compte.
Fixez-vous un objectif d'épargne précis et une date pour l'atteindre.
Combien dois-je épargner chaque mois ?
Il n'existe pas de règle unique : épargner régulièrement, même une petite somme, compte davantage que le montant.
Avant de demander un prêt, calculez le coût total du crédit et pas seulement la mensualité.
L'inflation réduit la valeur de l'argent qui dort : une épargne rémunérée aide à préserver votre pouvoir d'achat.
Microfinance
Établissement de microfinance (EMF)
Microcrédit
Crédit agricole de campagne
Crédit solidaire
Le microcrédit de {montant} FCFA a été accordé à votre groupe.
Un agriculteur peut-il obtenir un crédit pour la saison des semences ?
Un collecteur passe chaque jour recueillir votre épargne, même de petits montants, et vous la retrouvez sur votre compte.
Oui, le crédit de campagne finance les intrants et se rembourse après la récolte.
Dans le crédit solidaire, les membres du groupe se portent garants les uns des autres pour le remboursement.
Les établissements de microfinance de la zone CEMAC sont agréés et contrôlés par la COBAC.
Le crédit de campagne doit être remboursé au plus tard le {date}, après la vente de la récolte.
L'assurance-vie vous permet de constituer une épargne tout en protégeant vos proches en cas de décès.
Pour acheter du crédit pour un proche, choisissez « Achat de crédit », puis « Pour un autre numéro ».
Refusez : prêter son compte pour faire transiter de l'argent d'inconnus peut vous rendre complice d'une escroquerie ou de blanchiment.
Puis-je avoir un compte dans une banque et un autre dans une microfinance ?
Le chèque n° {numero} de {montant} FCFA a été crédité sur votre compte.
Votre attestation de domiciliation bancaire est prête. Vous pouvez la télécharger dans votre espace client.
Total des débits du mois : {montant} FCFA — Total des crédits du mois : {montant} FCFA
Conservez le ticket et déposez immédiatement une réclamation ; la différence sera recréditée après le contrôle du distributeur.
Que se passe-t-il si je perds mon emploi pendant que je rembourse mon crédit ?
Pourquoi ma demande de prêt a-t-elle été refusée ?
Puis-je demander un prêt par Mobile Money sans aller en agence ?
Mon conjoint doit-il signer le contrat de prêt ?
Si vous avez souscrit une assurance perte d'emploi, elle peut prendre en charge une partie des mensualités. Sinon, contactez rapidement votre conseiller.
Certains opérateurs proposent des micro-prêts directement sur le téléphone, avec un montant et une durée limités.
Le nombre de mensualités restantes figure dans votre échéancier, disponible dans l'application ou en agence.
Votre prêt de {montant} FCFA a été versé sur votre compte. Montant à rembourser avant le {date} : {montant} FCFA.
Votre prêt arrive à échéance demain. Assurez-vous d'avoir {montant} FCFA sur votre compte pour le remboursement automatique.
Votre dossier de crédit est complet et a été transmis au comité d'engagement. Réponse attendue sous {duree}.
Il manque votre attestation de travail pour finaliser votre demande de prêt.
Vous n'êtes pas encore éligible à un prêt. Continuez à utiliser votre compte régulièrement.
Vous avez un prêt en cours non remboursé. Remboursez-le avant de faire une nouvelle demande.
Ne signez pas l'offre de prêt avant d'avoir vérifié le montant total à rembourser.
Un prêt non remboursé à temps peut réduire vos possibilités d'emprunt à l'avenir.
Emprunter pour rembourser un autre crédit peut vous faire entrer dans une spirale d'endettement.
Le prêt immobilier finance l'achat d'un terrain, la construction ou la rénovation d'un logement, sur une durée pouvant aller jusqu'à plusieurs années.
Le crédit de trésorerie aide une entreprise à faire face à un décalage entre ses dépenses et ses encaissements.
Comment épargner un peu chaque jour avec mon téléphone ?
Est-ce plus intéressant de garder mon argent sur mon compte courant ou sur un compte d'épargne ?
Puis-je ouvrir un compte d'épargne pour les études de mon enfant ?
Mon épargne est-elle garantie si la banque fait faillite ?
Activez un versement automatique depuis votre portefeuille mobile vers votre compte d'épargne, même pour de petites sommes.
Oui, un compte d'épargne peut être ouvert au nom d'un enfant mineur par l'un de ses parents.
Il vous reste {montant} FCFA à épargner pour atteindre votre objectif avant le {date}.
{montant} FCFA seront transférés vers votre épargne le {date} de chaque mois.
Le coupon de votre obligation {nom}, soit {montant} FCFA, a été crédité le {date}.
Je vends au marché depuis cinq ans. Quels documents la microfinance va-t-elle me demander pour un crédit ?
Combien de membres faut-il pour former un groupe de crédit solidaire ?
Puis-je rembourser mon crédit agricole en une seule fois après la récolte ?
Oui, certains crédits de campagne prévoient un remboursement unique après la vente de la récolte.
Votre versement de {montant} FCFA sur votre compte d'épargne a été enregistré par l'agence {agence}.
Rappel : la réunion de votre groupe de crédit solidaire aura lieu le {date} à {heure}.
Les coopératives agricoles peuvent obtenir des crédits collectifs pour acheter des équipements que chaque producteur ne pourrait pas financer seul.
Chaque membre du groupe est solidairement responsable du remboursement de la totalité du crédit accordé au groupe.
L'assurance emprunteur rembourse tout ou partie de votre crédit en cas de décès ou d'invalidité, pour ne pas laisser de dette à vos proches.
Les dépenses des fêtes, de la rentrée scolaire et des événements familiaux se prévoient à l'avance : épargnez un peu chaque mois pour ne pas avoir à emprunter.
Un fonds d'urgence évite de recourir à un crédit coûteux lorsqu'un imprévu survient, comme une maladie ou une panne.
Épargnez d'abord, dépensez ensuite : programmez votre épargne le jour où vous recevez votre revenu.
Ne prêtez que l'argent que vous pouvez vous permettre de ne pas récupérer.
Achat de crédit téléphonique
Remboursement de prêt
Crédit
Afficher uniquement les crédits
Épargne personnelle
Épargne mobile
Achat de crédit de {montant} FCFA effectué pour le {numero_telephone}.
Votre épargne mobile a été créditée de {montant} FCFA.
La tontine numérique vous permet d'épargner avec votre groupe. Chaque membre verse sa cotisation à date fixe et la somme collectée est remise à tour de rôle.
Votre achat de crédit a été débité mais le crédit n'est pas arrivé ? Patientez 15 minutes puis contactez le service client avec la référence {reference}.
Numéro de téléphone
Note (optionnel)
Montant
Total débité
Téléphone
Référence
Opération
Je confirme que les informations ci-dessus sont correctes.
Modifier
Statut
Succès
Bonjour, {nom}
Portefeuille électronique
Agent agréé
Forfait internet
Services bancaires
0. Retour
00. Menu principal
Annuler l'opération
Scanner le QR code
Choisir dans mes contacts
Recharger mon portefeuille
Numéro du destinataire
Numéro de l'agent
Que faire si j'ai envoyé de l'argent au mauvais numéro ?
Que faire si l'agent n'a pas assez de liquidités ?
Saisissez le numéro du destinataire sans l'indicatif du pays.
Vérifiez le nom du destinataire avant de confirmer.
Ne remettez votre téléphone à personne pendant l'opération.
Évitez les codes comme 0000, 1234 ou votre année de naissance.
Répondez avec le chiffre correspondant à votre choix.
Nouveau : payez vos achats en ligne avec votre portefeuille mobile.
Le numéro saisi n'est pas valide.
Le montant minimum autorisé est de {montant} FCFA.
Le délai de confirmation a expiré. Veuillez recommencer l'opération.
Service momentanément indisponible. Réessayez plus tard.
L'agent n'a pas validé l'opération. Aucun montant n'a été débité.
Veuillez saisir un montant.
Méfiez-vous des messages qui vous annoncent un gain à une loterie à laquelle vous n'avez pas participé.
Un vrai agent ne vous demande jamais de composer un code sur votre propre téléphone pour débloquer une somme.
Montant total débité : {montant} FCFA
Bienvenue sur votre portefeuille mobile
Récapitulatif de l'opération
Opération réussie
Trouver un agent près de chez vous
Donneur d'ordre
Code SWIFT
IBAN
Code banque
Code guichet
Clé RIB
Date d'exécution
Pays de destination
Devise de réception
En attente
Exécuté
Rejeté
Remboursé
Modifier le montant
Vérifier les informations
Je confirme
Date d'exécution souhaitée
Fréquence (hebdomadaire, mensuelle)
Montant à recevoir
Saisissez les 23 caractères du RIB sans espaces.
Le RIB saisi est incorrect. Vérifiez la clé RIB.
Le code SWIFT saisi n'existe pas.
La date d'exécution ne peut pas être antérieure à aujourd'hui.
Pays de destination non desservi.
La banque n'est pas responsable des retards dus à une erreur dans les coordonnées fournies par le client.
Date d'exécution prévue : {date}
Recevoir de l'argent de la diaspora
Envoyez de l'argent vers le Gabon, le Congo, le Tchad, la Centrafrique ou la Guinée équatoriale sans changer de devise, puisque le FCFA est commun aux pays de la CEMAC.
Hameçonnage
Authentification à deux facteurs
Ingénierie sociale
Empreinte digitale
"""   # <- remplacer par vos phrases, une par ligne

def tester(fr, modele_seul=None, beams=BEAMS, synonymes=None):
    t = translate(fr, modele_seul=MODELE_SEUL if modele_seul is None else modele_seul, synonymes=synonymes, beams=beams)
    return {'Français': norm_fr(fr), 'Traduction': t['dua'], 'Source': t['source'],
            'Synonymes': ', '.join(f'{a} -> {b}' for a, b in t['subs']), 'Mots inconnus': ', '.join(t['inconnus']),
            'Score': t['score'], 'À relire': ' ; '.join(t['flags']), 'Sortie brute': t['brut']}

lignes = [l.strip() for l in MES_PHRASES.strip().splitlines() if l.strip()]
res_perso = pd.DataFrame([tester(l) for l in lignes])
pd.set_option('display.max_colwidth', 80, 'display.width', 250)
display(res_perso) if 'display' in globals() else print(res_perso.to_string())


                                                                                                                                                                     Français                                                                                                                                                              Traduction  Source                                                                            Synonymes                                                                             Mots inconnus  Score                                                                                           À relire                                                                                                                                                            Sortie brute
0                                                                                                                                                                       Solde                            

## 8. Sauvegarde pour l'extension
Le dossier `duala_model` (≈2,4 Go) + `glossaire.json` (le remplacement par synonymes s'appuie sur WordNet ) se copient sur Google Drive, ou se servent via une petite API (FastAPI) appelée par l'extension.

In [15]:
model.save_pretrained('duala_model'); tok.save_pretrained('duala_model')
json.dump(dict(glossaire=gloss, natif=native, tons=tone_map, ratio=RATIO, score_min=SCORE_MIN, min_syn=MIN_SYN, max_len=MAX_LEN), open('duala_model/glossaire.json', 'w'), ensure_ascii=False)
pd.DataFrame(hist).to_csv('duala_model/courbes.csv', index=False)

# ---- meilleurs paramètres et performances de CET entraînement (à garder pour comparer les essais) ----
_b = {k: (None if isinstance(v, float) and v != v else v) for k, v in pd.DataFrame(hist).sort_values('critere').iloc[-1].to_dict().items()}
meilleurs = dict(
    date=time.strftime('%Y-%m-%d %H:%M'), duree_entrainement_min=TRAIN_MINUTES,
    parametres=dict(MODEL=MODEL, EPOCHS=EPOCHS, PATIENCE=PATIENCE, BS=BS, LR=LR, LS=LS, DROPOUT=DROPOUT, FREEZE_EMB=FREEZE_EMB, STRIP_TONES=STRIP_TONES,
                    MAX_LEN=MAX_LEN, MAX_TOK=MAX_TOK, NO_REPEAT=NO_REPEAT, VALIDATED_UPSAMPLE=VALIDATED_UPSAMPLE, EVAL_N=EVAL_N, EVAL_BEAMS=EVAL_BEAMS,
                    MAX_WORDS_DICO=MAX_WORDS_DICO, RATIO_ALIGN=RATIO_ALIGN, SEED=SEED),
    donnees=dict(paires_entrainement=len(train_ex), termes_val=len(val), termes_test=len(test), phrases_train=len(sent_tr), phrases_val=len(sent_va), phrases_test=len(sent_te)),
    meilleure_epoque_validation=_b,        # choisie sur la VALIDATION (jamais sur le test)
    test=globals().get('RES_TEST'))        # mesuré une seule fois, cellule 6
json.dump(meilleurs, open('duala_model/meilleurs_parametres.json', 'w'), ensure_ascii=False, indent=2, default=float)
_row = dict(date=meilleurs['date'], critere_val=_b['critere'], chrF_val=_b['chrF'], exact_val=_b['exact_pct'], epoque=_b['epoch'],
            chrF_test=(meilleurs['test'] or {}).get('dictionnaire', {}).get('chrF'), **{k: v for k, v in meilleurs['parametres'].items() if k in ('LR', 'BS', 'EPOCHS', 'DROPOUT', 'LS', 'FREEZE_EMB', 'STRIP_TONES')})
_h = pd.DataFrame([_row]); _f = 'duala_model/historique_essais.csv'
if os.path.exists(_f): _h = pd.concat([pd.read_csv(_f), _h])        # une ligne par entraînement : l'essai au meilleur critere_val = les meilleurs paramètres
_h.to_csv(_f, index=False); print(_h.to_string(index=False))
!zip -qr duala_model.zip duala_model
try: files.download('duala_model.zip')
except Exception as e: print('copier duala_model.zip sur Drive :', e)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

            date  critere_val  chrF_val  exact_val  epoque  chrF_test  EPOCHS  BS     LR  LS  DROPOUT  FREEZE_EMB  STRIP_TONES
2026-11-14 10:35         30.0      30.0       17.0      11      29.44      14  32 0.0001 0.1      0.2        True         True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/duala
!cp -r /content/duala_model /content/drive/MyDrive/duala/
!ls -lh /content/drive/MyDrive/duala/duala_model

In [17]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
from transformers import AutoModelForSeq2SeqLM

src = '/content/drive/MyDrive/duala/duala_model'
dst = '/content/drive/MyDrive/duala/duala_model_fp16'

m = AutoModelForSeq2SeqLM.from_pretrained(src).half()
m.save_pretrained(dst)                      # poids en 16 bits (~1,2 Go)
for f in os.listdir(src):                   # copie tokenizer, glossaire.json, etc.
    p = os.path.join(src, f)
    if os.path.isfile(p) and not f.startswith(('model', 'pytorch_model')) and not os.path.exists(os.path.join(dst, f)):
        shutil.copy(p, dst)
print(sum(os.path.getsize(os.path.join(dst, f)) for f in os.listdir(dst)) / 1e9, 'Go')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

1.263094716 Go


In [18]:
model.half()
print(evaluate(test)[0])    # à comparer à chrF 29.44 / exact 17 % de votre test d'origine
model.float()

{'BLEU': 16.57, 'chrF': np.float64(29.46), 'exact_pct': np.float64(15.64)}


M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(256206, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100Attention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=Tr